# Mistake-detection parameter sweeps — pooled injections across 195 source tracks

Recompute reports from completed prediction caches only: **195 distinct source tracks, two cached E2 injections per track (seeds 0 and 1, 25% mistake rate)**. Clean controls are excluded from every displayed metric. Original caches and two-seed reports remain unchanged.

For paired notes, cost is alpha × pitch distance + beta × duration distance. For an unmatched note of duration d, cost is rho + beta × d. The onset/duration balance is 0/1. These are parameter-tuning results, not held-out validation.

The completed mistake-only cohort correction (one horn replacement) is retained. Missing or inconsistent caches raise an error; Run All never renders audio, detects pitches/notes, or reruns alignment.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'app.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from benchmarks.notebooks.sweeps.scripts import mistake_cached_reporting as reporting
import importlib
reporting = importlib.reload(reporting)
OUTPUT = ROOT / 'benchmarks/results/sweeps/balanced_212_v3'
INJECTION_SEEDS = (0, 1)  # Fixed across all tracks/candidates; never selected by score.


## Protocol and provenance

Use the corrected 195-track Coco cohort (15 per instrument). Retain both cached injection seeds (0 and 1) at the 25% mistake rate. Pool missed/extra event counts across these 390 performances; substitutions count as missed + extra. Event matching uses 100 ms / 50 cents. Clean controls do not contribute to reports.

Cached predictions already include the selected pitch/note frontend, alignment, timing refitting, and repeat recovery. The reader checks cohort/frontend contracts, candidate parameters, complete 195-track coverage, and exactly one retained case per seed/track/candidate. Rankings and the selected configuration are recalculated for this subset.

In [ ]:
reports = reporting.load_reports(OUTPUT, seeds=INJECTION_SEEDS)
previous = reports['weights']
print('Weight prediction cache:', previous['source'])
print('Distinct source tracks: 195; performances: 390; injection seeds:', INJECTION_SEEDS)


## 1. Cached pitch/timing sweep — rho fixed at 5

Pitch weights: 0, 1, 2, 3, 4, 6, 8. Timing weights: 0, 0.5, 1, 2, 4. Exclude 0:0. Reaggregate **34 × 195 = 6,630** retained track/candidate case scores without running the pipeline.

In [ ]:
display(previous['summary'])


## 2. Cached fixed-penalty sensitivity sweep at 6:2

Reaggregate the completed rho = 3, 4, 5, 6, 7 candidates at alpha = 6 and beta = 2 using the same 390 performances across 195 distinct source tracks. These weights were fixed for the original conditional follow-up; this is not a joint search over all weights and penalties.

In [ ]:
penalty_results = reports['penalties']
display(penalty_results['summary'])
print('Penalty prediction cache:', penalty_results['source'])


## 3. Combined ranking for the 195-source-track subset

Deduplicate the shared 6:2, rho=5 candidate and rank all 38 distinct cached configurations by pooled mistake-event F1. Both original injection seeds contribute to the ranking. No production settings or historical reports are overwritten.

In [ ]:
summary = reports['summary']
display(summary.head(5))
winners = summary[summary['Mistake-Event F1'] == summary['Mistake-Event F1'].max()]
print('Configurations tied for highest F1:')
display(winners)
# Retain the previous choice if it is still tied for best; otherwise use
# the report's deterministic alpha/beta/rho ordering to resolve an exact tie.
retained = winners[(winners.alpha == 6) & (winners.beta == 2) & (winners.rho == 5)]
best = (retained if not retained.empty else winners).iloc[0]
SELECTED = dict(alignment_gamma_pitch=float(best['alpha']),
                alignment_gamma_time=float(best['beta']),
                ins_cost=float(best['rho']), del_cost=float(best['rho']))
print('Selected on 390 performances across 195 distinct source tracks:', SELECTED)
